# Running MACH2, MACHINA and Metient

This notebook runs the methods compared in the paper on the instances in `../data` (see Section 3.1 of the README).
1. generates the **MACHINA** input (and MACHINA-format ground truth) with `PullDownLeaves`,
2. generates the **Metient** input,
3. runs **MACH2**, **MACHINA** and **Metient** with the settings of the paper.

Generated inputs are written to `inputs/` and results to `results/<instance>/<method>/`
(method = `MACH2-<ordering>`, `MACHINA`) and `results/metient/<cohort>/`.

Sections are independent functions; run the setup cell, then only what you need.
Requirements: `mach2` + Gurobi (§3), MACHINA's `pmh_tr` (§4), `metient` (§5). §1–2 need only Python.

In [ ]:
import glob, itertools, os, shutil, subprocess

DATA = os.path.abspath('../data')        # MACH2 inputs
INPUTS = os.path.abspath('inputs')       # generated MACHINA / Metient inputs
RESULTS = os.path.abspath('results')     # where the runs write

def edges(f): return [tuple(l.split()) for l in open(f) if l.strip()]
def labeling(f): return {l.split()[0]: l.split()[1:] for l in open(f) if l.strip()}
def nodes(E): return list(dict.fromkeys(n for e in E for n in e))
def write(f, pairs, sep='\t'):
    os.makedirs(os.path.dirname(f), exist_ok=True)
    open(f, 'w').write(''.join(f'{a}{sep}{b}\n' for a, b in pairs))

def file(prefix, suffix):                # sims/<...>/<seed>/<suffix>, real/<cohort>/<id>.<suffix>
    return f"{prefix}/{suffix}" if prefix.startswith('sims/') else f"{prefix}.{suffix}"
def path(r, suffix):                     # files in ../data (mach2.*, gt.mach2.*, mutations.tsv)
    return f"{DATA}/{file(r['prefix'], suffix)}"
def generated(r, suffix):                # files generated in §1–2 (machina.*, gt.machina.*, metient.*)
    return f"{INPUTS}/{file(r['prefix'], suffix)}"
def outdir(method, r, p):                # results/<instance>/<method>/[<primary>/]
    d = os.path.join(RESULTS, r['prefix'], method, p if len(r['primary']) > 1 else '')
    os.makedirs(d, exist_ok=True); return d

def primary_only(prefix):                # set 1: every migration of the ground truth leaves the primary
    E, lab = edges(f"{DATA}/{file(prefix, 'gt.mach2.tree')}"), labeling(f"{DATA}/{file(prefix, 'gt.mach2.labeling')}")
    return all(lab[u] == lab[v] or lab[u] == ['P'] for u, v in E)

def instance(prefix):
    parts = prefix.split('/')
    dataset, group, inst = parts[1], '/'.join(parts[2:-1]), parts[-1]
    if parts[0] == 'sims':
        primary = ['P']
        cohort = {'set1': f"set1/{'primary_only' if primary_only(prefix) else 'met_to_met'}",
                  'set2': f'set2/{group}'}.get(dataset, '')      # Metient was not run on set 3
    elif dataset == 'lung':
        primary, cohort = (['p.lung'], 'lung/CRUK0063') if inst == 'CRUK0063_1' else (['Lung'], 'lung')
    elif dataset == 'ovarian':
        primary, cohort = (['LOv', 'ROv'] if inst in {'1', '3', '4', '7', '9'} else ['ROv']), 'ovarian'
    else:                                # prostate, breast
        primary, cohort = [dataset], dataset
    return dict(dataset=dataset, group=group, instance=inst, prefix=prefix, primary=primary, metient_cohort=cohort)

rows = [instance(p) for p in sorted(
    [os.path.relpath(os.path.dirname(f), DATA) for f in glob.glob(f'{DATA}/sims/**/mach2.tree', recursive=True)] +
    [os.path.relpath(f, DATA)[:-len('.mach2.tree')] for f in glob.glob(f'{DATA}/real/**/*.mach2.tree', recursive=True)])]

def select(*datasets):                   # instances of the given datasets (all if none given)
    return [r for r in rows if not datasets or r['dataset'] in datasets]

len(rows)

## 1. MACHINA input

`PullDownLeaves` (Supplementary Algorithm 1): a leaf observed in exactly one location keeps that location;
for every other observed (clone `v`, location `s`) pair a leaf `v_s` labeled `s` is attached to `v`.
Writes `machina.tree` / `machina.labeling`, and for simulations the ground truth
`gt.machina.tree` / `gt.machina.labeling` (true location of every vertex).

In [ ]:
def pull_down_leaves(E, lab):
    inner, E2, leaf = {u for u, _ in E}, list(E), {}
    for v in nodes(E):
        locs = lab.get(v, [])
        if v not in inner and len(locs) == 1:
            leaf[v] = locs[0]
        else:
            for s in locs:
                E2.append((v, f'{v}_{s}')); leaf[f'{v}_{s}'] = s
    return E2, leaf

def to_machina(r):
    E, leaf = pull_down_leaves(edges(path(r, 'mach2.tree')), labeling(path(r, 'mach2.labeling')))
    write(generated(r, 'machina.tree'), E); write(generated(r, 'machina.labeling'), leaf.items())
    if os.path.exists(path(r, 'gt.mach2.labeling')):
        gt = labeling(path(r, 'gt.mach2.labeling'))
        write(generated(r, 'gt.machina.tree'), E)
        write(generated(r, 'gt.machina.labeling'), [(v, leaf.get(v) or gt[v][0]) for v in nodes(E)])

In [ ]:
for r in select(): to_machina(r)

## 2. Metient input

One `metient.tsv` (clone × site presence, site category, #mutations) and `metient.tree`
(edges between clone indices). #mutations per clone: `mutations.tsv` for real data (the counts given to
Metient in the paper); for simulations the number of driver mutations in the clone name, as in the original runs.
Not generated for set 3 (Metient was not run on it).

In [ ]:
def num_mutations(r, V):
    if os.path.exists(path(r, 'mutations.tsv')):
        return dict(l.split() for l in list(open(path(r, 'mutations.tsv')))[1:])
    return {u: 0 if u == 'GL' else 1 if '_' in u else len(u.split(';')) for u in V}

def to_metient(r):
    E, lab = edges(path(r, 'mach2.tree')), labeling(path(r, 'mach2.labeling'))
    V = nodes(E); nm = num_mutations(r, V); idx = {u: i for i, u in enumerate(V)}
    sites = sorted({s for l in lab.values() for s in l})
    table = [(i, s, idx[u], u, int(s in lab.get(u, []) or (u == 'GL' and s == 'P')),
              'primary' if s in r['primary'] else 'metastasis', nm[u]) for i, s in enumerate(sites) for u in V]
    os.makedirs(os.path.dirname(generated(r, 'metient.tsv')), exist_ok=True)
    with open(generated(r, 'metient.tsv'), 'w') as f:
        f.write('anatomical_site_index\tanatomical_site_label\tcluster_index\tcluster_label\tpresent\tsite_category\tnum_mutations\n')
        f.writelines('\t'.join(map(str, t)) + '\n' for t in table)
    write(generated(r, 'metient.tree'), [(idx[u], idx[v]) for u, v in E], sep=' ')

In [ ]:
for r in select():
    if r['metient_cohort']: to_metient(r)

## 3. Run MACH2

Criteria orderings and solution limits from Methods. Set 1: the eleven orderings below (the other orderings
are derived from these solution spaces); set 2: all 24 orderings of UMCS; set 3: five orderings; real data: UMCS
(each ovarian primary separately). Output: `results/<instance>/MACH2-<ordering>/`.

In [ ]:
MACH2_ORDERINGS = {'set1': ['UM', 'UC', 'USM', 'USC', 'M', 'CU', 'CM', 'CS', 'SU', 'SM', 'SC'],
                   'set2': [''.join(p) for p in itertools.permutations('UMCS')],
                   'set3': ['UMCS', 'MUCS', 'MCUS', 'MCSU', 'UMSC']}   # real data: UMCS

def run_mach2(r, ordering, threads=0):
    import mach2
    tree = mach2.MultiLabeledTree.from_files(path(r, 'mach2.tree'), path(r, 'mach2.labeling'))
    max_solutions = 37888 if ordering == 'M' else 8192
    for p in r['primary']:
        sols = mach2.MACH2(tree, primary_location=p, criteria_ordering=ordering).solve(max_solutions=max_solutions, threads=threads)
        sols.write(outdir(f'MACH2-{ordering}', r, p) + '/')

def run_mach2_all(*datasets):
    for r in select(*datasets):
        for o in MACH2_ORDERINGS.get(r['dataset'], ['UMCS']):
            run_mach2(r, o)

In [ ]:
# run_mach2(select('breast')[0], 'UMCS')   # one instance
# run_mach2_all('prostate', 'breast')      # whole datasets (set 3 takes days)

## 4. Run MACHINA

Needs MACHINA's `pmh_tr` (build from <https://github.com/raphael-group/machina>; requires LEMON and Gurobi)
and the inputs of §1. Settings of the original runs: all migration patterns (`-m 3`: PS, S, M, R), 30 threads.
MACHINA requires a colour map; one is written to the output folder.
`write_nodes` adds the node-of-origin file needed to load MACHINA output with `mach2.Refinement.from_files`.

In [ ]:
PMH_TR = 'pmh_tr'   # path to the MACHINA binary

def coloring(locs, out):
    write(f'{out}/coloring.txt', [(s, i + 1) for i, s in enumerate(sorted(locs))])
    return f'{out}/coloring.txt'

def run_machina(r, threads=30):
    locs = {v[0] for v in labeling(generated(r, 'machina.labeling')).values()}
    for p in r['primary']:
        out = outdir('MACHINA', r, p)
        cmd = [PMH_TR, generated(r, 'machina.tree'), generated(r, 'machina.labeling'), '-p', p,
               '-c', coloring(locs, out), '-m', '3', '-t', str(threads), '-o', out]
        with open(f'{out}/log.txt', 'w') as log: subprocess.run(cmd, check=True, stdout=log)
        write_nodes(r, out)

def write_nodes(r, out):   # node of origin of every MACHINA output node (loads with mach2.Refinement.from_files)
    V = set(nodes(edges(path(r, 'mach2.tree'))))
    for t in glob.glob(f'{out}/*-T-*.tree'):
        E = edges(t); par = {v: u for u, v in E}
        def origin(u): return u if u in V else u.split('^')[0] if '^' in u else origin(par[u])
        write(t[:-5] + '.nodes', [(u, origin(u)) for u in nodes(E)])
    for f in glob.glob(f'{out}/*.dot'): os.remove(f)

In [ ]:
# for r in select('prostate'): run_machina(r)

## 5. Run Metient

`calibrate_label_clone_tree` once per cohort (set 1 split into primary-only and metastasis-to-metastasis,
set 2 per condition, one cohort per real cancer type, CRUK0063 on its own).
Settings of the original runs (the Methods text states 8192 samples for all; the simulation runs used
`k_best_trees=1500`). Breast additionally uses organotropism from the MSK-MET metastasis frequencies
(`MSK_MET_FREQ`, the `msk_met_freq_by_cancer_type.csv` table distributed with Metient).
Output: `results/metient/<cohort>/`.

In [ ]:
MSK_MET_FREQ = 'msk_met_freq_by_cancer_type.csv'   # path to the MSK-MET table (breast organotropism)

METIENT = {'set1': dict(k_best_trees=1500, solve_polytomies=False),
           'set2': dict(k_best_trees=1500, solve_polytomies=False),
           'prostate': dict(k_best_trees=8192, solve_polytomies=True, sample_size=8192),
           'ovarian': dict(k_best_trees=8192, solve_polytomies=True, sample_size=8192),
           'lung': dict(k_best_trees=8192, solve_polytomies=True, sample_size=8192),
           'breast': dict(k_best_trees=8192, solve_polytomies=True, organotropism=True)}
BREAST_SITES = {'liver': 'Liver', 'brain': 'CNS/Brain', 'rib': 'Bone', 'breast': 'Breast', 'kidney': 'Kidney',
                'lung': 'Lung', 'adrenal': 'Adrenal Gland', 'spinal': 'CNS/Brain'}

def organotropism(r):
    from metient.util import data_extraction_util as dutil
    sites = sorted({s for l in labeling(path(r, 'mach2.labeling')).values() for s in l})   # order of metient.tsv
    return dutil.get_organotropism_matrix_from_msk_met(sites, 'Breast Cancer', MSK_MET_FREQ, BREAST_SITES)

def run_metient(cohort):
    from metient import metient as met
    R = [r for r in rows if r['metient_cohort'] == cohort]
    cfg = dict(METIENT[R[0]['dataset']]); k = cfg.pop('k_best_trees'); ot = cfg.pop('organotropism', False)
    if ot: cfg['Os'] = [organotropism(r) for r in R]
    out = f'{RESULTS}/metient/{cohort}/'; os.makedirs(out, exist_ok=True)
    met.calibrate_label_clone_tree([generated(r, 'metient.tree') for r in R], [generated(r, 'metient.tsv') for r in R],
                                   met.PrintConfig(visualize=False, verbose=False, k_best_trees=k), out,
                                   ['_'.join(r['prefix'].split('/')[-2:]) for r in R], 'both' if ot else 'genetic', **cfg)

def run_metient_all():
    for c in sorted({r['metient_cohort'] for r in rows} - {''}): run_metient(c)

In [ ]:
# run_metient('prostate')

## 6. Checks and clean-up

`check_ground_truth` confirms that both ground-truth formats encode the same migrations (needs `mach2`);
`clean` removes the files generated in §1–2.

In [ ]:
def check_ground_truth(r):
    import mach2
    mg = lambda *f: sorted(mach2.Refinement.from_files(*f).migration_graph().get_each_migration().items())
    return (mg(*(generated(r, s) for s in ['gt.machina.tree', 'gt.machina.labeling', 'machina.tree', 'machina.labeling'])) ==
            mg(*(path(r, s) for s in ['gt.mach2.tree', 'gt.mach2.labeling', 'mach2.tree', 'mach2.labeling'])))

def clean():
    shutil.rmtree(INPUTS, ignore_errors=True)

In [ ]:
# all(check_ground_truth(r) for r in select('set1', 'set2'))
# clean()